# Observational Data Staging Workflow

This notebook stages the SEANOE mooring QC data release into the project data layout for a run-dated snapshot. It defines source links and output locations, downloads raw ZIP files, extracts them into staged folders, and performs a quick directory preview to confirm the expected files were staged.

In [1]:
# Cell 1: Configuration
from pathlib import Path
from datetime import date

DATASET_PAGE = "https://www.seanoe.org/data/01021/113246/"
DOI = "10.17882/113246"

FILES = {
    "ctd_mooring_qc": "https://www.seanoe.org/data/01021/113246/data/127701.zip",
    "adcp_mooring_qc": "https://www.seanoe.org/data/01021/113246/data/127702.zip",
}

RUN_DATE = date.today().isoformat()

BASE_DIR = Path.cwd() / "/anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data"
RAW_DIR = BASE_DIR / "raw" / "seanoe_113246" / RUN_DATE
STAGED_DIR = BASE_DIR / "staged" / "seanoe_113246" / RUN_DATE
MANIFEST_CSV = BASE_DIR / "staged" / "seanoe_113246" / f"manifest_{RUN_DATE}.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
STAGED_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("STAGED_DIR:", STAGED_DIR)

RAW_DIR: /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/raw/seanoe_113246/2026-04-27
STAGED_DIR: /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/staged/seanoe_113246/2026-04-27


## Step 1: Utility Functions

These helpers provide reusable functions for checksum generation, downloading files, and extracting ZIP archives.

In [2]:
# Cell 2: Helpers
import csv
import hashlib
import shutil
import urllib.request
import zipfile

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def download_file(url: str, dest: Path) -> Path:
    if dest.exists():
        print(f"Already exists, skipping download: {dest.name}")
        return dest

    print(f"Downloading: {url}")
    with urllib.request.urlopen(url) as response, dest.open("wb") as out:
        shutil.copyfileobj(response, out)
    print(f"Saved: {dest}")
    return dest

def unzip_file(zip_path: Path, extract_to: Path) -> list[Path]:
    extract_to.mkdir(parents=True, exist_ok=True)
    extracted = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
        extracted = [extract_to / name for name in zf.namelist()]
    return extracted

## Step 2: Download Raw Archives

This step fetches each configured source file into the run-specific raw data directory.

In [3]:
# Cell 3: Download raw ZIP files
downloaded = {}

for key, url in FILES.items():
    dest = RAW_DIR / f"{key}.zip"
    downloaded[key] = download_file(url, dest)

downloaded

Downloading: https://www.seanoe.org/data/01021/113246/data/127701.zip
Saved: /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/raw/seanoe_113246/2026-04-27/ctd_mooring_qc.zip
Downloading: https://www.seanoe.org/data/01021/113246/data/127702.zip
Saved: /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/raw/seanoe_113246/2026-04-27/adcp_mooring_qc.zip


{'ctd_mooring_qc': PosixPath('/anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/raw/seanoe_113246/2026-04-27/ctd_mooring_qc.zip'),
 'adcp_mooring_qc': PosixPath('/anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/raw/seanoe_113246/2026-04-27/adcp_mooring_qc.zip')}

## Step 3: Extract Into Staging

Downloaded archives are unpacked into staged subfolders so downstream processing can use normalized file locations.

In [4]:
# Cell 4: Extract ZIP files into staged folders
extracted_index = {}

for key, zip_path in downloaded.items():
    out_dir = STAGED_DIR / key
    extracted_files = unzip_file(zip_path, out_dir)
    extracted_index[key] = extracted_files
    print(f"{key}: extracted {len(extracted_files)} items to {out_dir}")

ctd_mooring_qc: extracted 19 items to /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/staged/seanoe_113246/2026-04-27/ctd_mooring_qc
adcp_mooring_qc: extracted 15 items to /anvil/projects/x-ees250129/x-uheede/C-Star-in-Hvalfjordur/data/staged/seanoe_113246/2026-04-27/adcp_mooring_qc


## Step 4: Quick Staging Preview

This quick check prints a small sample of staged paths to verify extraction results before moving on.

In [5]:
# Cell 6 (optional): quick preview of what was staged
from pathlib import Path

for group_dir in sorted(STAGED_DIR.iterdir()):
    if group_dir.is_dir():
        sample = list(group_dir.rglob("*"))[:5]
        print(f"\n{group_dir.name} (showing up to 5 paths):")
        for p in sample:
            print(" -", p.relative_to(STAGED_DIR))


adcp_mooring_qc (showing up to 5 paths):
 - adcp_mooring_qc/Mooring_data_Final_.nc
 - adcp_mooring_qc/Mooring_data_Final_.nc/HVNV3_ADCP_33m_20250217_20250428.nc
 - adcp_mooring_qc/Mooring_data_Final_.nc/HVSA2_ADCP_30m_20240814_20250217.nc
 - adcp_mooring_qc/Mooring_data_Final_.nc/HVIN2_ADCP_59m_20240814_20250217.nc
 - adcp_mooring_qc/Mooring_data_Final_.nc/HVNV2_ADCP_33m_20240815_20250116.nc

ctd_mooring_qc (showing up to 5 paths):
 - ctd_mooring_qc/Mooring data final_.csv
 - ctd_mooring_qc/Mooring data final_.csv/HVNA2_CTDox_24m_20240814_20250217.csv
 - ctd_mooring_qc/Mooring data final_.csv/HVSA3_CTDox_27m_20250217_20250430.csv
 - ctd_mooring_qc/Mooring data final_.csv/HVIN1_CTDphox_60m_20240404_20240814.csv
 - ctd_mooring_qc/Mooring data final_.csv/HVIN2_CTDphox_57m_20240814_20250217.csv


In [ ]:
## Notebook Status

Staging is complete for this run date, and the configured staged directory now contains extracted observation files ready for downstream processing.